# 08 — End-to-End Pipeline

**Goal:** Run the complete experiment using `src/run_experiment.py` — from raw CSV to final test evaluation — inspect all generated artifacts, and understand how every preceding notebook fits into the orchestration.

**Module:** `src/run_experiment.py`

---

## 0 · Imports

In [ ]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.utils.config import load_config

CONFIG_PATH = ROOT / 'configs' / 'base.yaml'
config = load_config(str(CONFIG_PATH))
print('Config loaded successfully.')
print(f'Experiment name : {config["experiment"]["name"]}')
print(f'Enabled models  : {config["models"]["enabled"]}')

---
## 1 · Pipeline architecture overview

```
run_experiment.py  ─  main(config_path)
│
├─ 1. load_config()                  ← configs/base.yaml
├─ 2. set_global_seed(42)
├─ 3. load_market_data()             ← data/raw/*.csv
├─ 4. apply_missing_value_policy()   ← ffill_then_drop_head
├─ 5. select_columns()               ← 7 features + close + date
├─ 6. split_dev_test()               ← 85% dev / 15% test
│
├─ For each (model, scaler) pair:
│   └─ run_cv_for_model_and_scaler()
│       ├─ make_expanding_folds()    ← 4 folds
│       └─ For each fold:
│           ├─ compute_forward_return()
│           ├─ compute_threshold()   ← train only
│           ├─ make_labels()
│           ├─ build_sequences()     ← lookback=21
│           ├─ drop_neutral_sequences()
│           ├─ scale_sequence_data() ← fit on train only
│           ├─ build_model()
│           └─ run_single_{sklearn|torch}_fold()
│
├─ select_best_cv_result()           ← by MCC mean
│
├─ run_final_train_and_test()        ← train on full dev, eval on test
│
└─ Save artifacts:
    ├─ artifacts/metrics/{name}_report.json
    ├─ artifacts/predictions/{name}_test_predictions.csv
    └─ experiments_summary.csv
```

---
## 2 · Enable the models you want to run

Edit `configs/base.yaml` → `models.enabled` to select which models to train. By default only `logreg` and `gru` are enabled to keep runtime short.

In [ ]:
print('Currently enabled models:')
for m in config['models']['enabled']:
    print(f'  • {m}')

print('\nAll available models:')
all_models = ['logreg', 'rf', 'xgboost', 'lightgbm', 'catboost',
              'gru', 'cnn1d', 'lstm', 'tcn', 'transformer_encoder']
for m in all_models:
    status = '✓ enabled' if m in config['models']['enabled'] else '  disabled'
    print(f'  {status}  {m}')

---
## 3 · Run the experiment

**Option A — from a terminal:**

```bash
cd <project_root>
python -m src.run_experiment --config configs/base.yaml
```

**Option B — from this notebook:**

In [ ]:
# Uncomment and run when you're ready to execute the full pipeline.
# Warning: this can take several minutes depending on enabled models.

# from src.run_experiment import main
# main(str(CONFIG_PATH))

print('Pipeline not yet executed — uncomment the lines above to run.')

---
## 4 · Inspect generated artifacts

After running the pipeline the following files are created:

In [ ]:
artifacts_root = ROOT / 'artifacts'

for subdir in ['metrics', 'predictions', 'models', 'plots']:
    folder = artifacts_root / subdir
    files  = list(folder.iterdir()) if folder.exists() else []
    print(f'artifacts/{subdir}/  ({len(files)} files)')
    for f in files:
        print(f'  {f.name}')

---
## 5 · Load and display the experiment report

In [ ]:
metrics_dir = artifacts_root / 'metrics'
json_files  = list(metrics_dir.glob('*.json'))

if not json_files:
    print('No reports found. Run the pipeline first (Section 3).')
else:
    report_path = sorted(json_files)[-1]  # most recent
    print(f'Loading: {report_path.name}\n')

    with open(report_path) as f:
        report = json.load(f)

    print('=== CV Summary ===')
    cv = report.get('cv_summary', {})
    for k, v in cv.items():
        if isinstance(v, float):
            print(f'  {k:<30}: {v:.4f}')

    print('\n=== Test Set Results ===')
    tm = report.get('test_metrics', {})
    for k, v in tm.items():
        if isinstance(v, float):
            print(f'  {k:<30}: {v:.4f}')

---
## 6 · Load test predictions

In [ ]:
preds_dir  = artifacts_root / 'predictions'
pred_files = list(preds_dir.glob('*.csv'))

if not pred_files:
    print('No prediction files found. Run the pipeline first.')
else:
    preds_df = pd.read_csv(sorted(pred_files)[-1])
    print(f'Predictions shape: {preds_df.shape}')
    print(preds_df.head(10).to_string())

---
## 7 · Experiments summary CSV

Every run appends a row to `experiments_summary.csv`. This provides a quick overview of all experiments performed.

In [ ]:
summary_csv = ROOT / 'experiments_summary.csv'

if summary_csv.exists():
    summary_df = pd.read_csv(summary_csv)
    print(f'Experiments logged: {len(summary_df)}')
    summary_df.sort_values('cv_mcc_mean', ascending=False).head(10)
else:
    print('experiments_summary.csv not found yet — it is created on the first run.')

---
## 8 · Final test confusion matrix (when available)

In [ ]:
import seaborn as sns

if not json_files:
    print('Run the pipeline to generate a real confusion matrix.')
else:
    cm = report.get('test_metrics', {}).get('confusion_matrix')
    if cm:
        cm_arr = np.array(cm)
        labels = ['Bear (0)', 'Bull (1)']
        fig, ax = plt.subplots(figsize=(5, 4))
        sns.heatmap(cm_arr, annot=True, fmt='d', cmap='Blues',
                    xticklabels=labels, yticklabels=labels,
                    linewidths=0.5, ax=ax)
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True')
        ax.set_title('Final Test Set — Confusion Matrix', fontsize=12)
        plt.tight_layout()
        plt.show()
    else:
        print('Confusion matrix not found in report.')

---
## 9 · Notebook series recap

| Notebook | Focus |
|---|---|
| 01 | Data loading, chronological order, feature visualisation |
| 02 | Preprocessing, forward returns, threshold, Bull/Bear labeling |
| 03 | Dev/test split, expanding CV folds, 3-D sequence tensors |
| 04 | Logistic Regression + 4 tree ensembles through CV |
| 05 | GRU / CNN1D / LSTM / TCN / Transformer architecture walkthrough |
| 06 | Deep learning training loop, LR search, early stopping |
| 07 | Metrics, confusion matrices, multi-model comparison |
| **08** | **End-to-end orchestration — `run_experiment.py`** |

---

**EHB 420E — Artificial Neural Networks, Istanbul Technical University, Spring 2026**